# Aufgabe 2: Ausreisser, Anomalien und Noise

Die vollständige Aufgabenstellung steht in [README.md](README.md). Ihr startet mit dem nach Aufgabe 1 veröffentlichten Checkpoint. Alle Ausgangszeilen und bisherigen Entscheidungen bleiben darin auditierbar.

**Leitplanke:** IQR und Isolation Forest erzeugen Untersuchungskandidaten, keine automatischen Wahrheitslabels.

## 0. Setup und Eingangskontrollen

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", context="notebook")

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        checkpoint = candidate / "data/checkpoints/checkpoint_01_resolved_events.parquet"
        zones = candidate / "data/reference/taxi_zone_lookup.csv"
        if checkpoint.exists() and zones.exists():
            return candidate
    raise FileNotFoundError(
        "Repository-Wurzel oder benötigte Dateien nicht gefunden. "
        "Wurde der erste Walkthrough bereits synchronisiert?"
    )

ROOT = find_repo_root()
CHECKPOINT_PATH = ROOT / "data/checkpoints/checkpoint_01_resolved_events.parquet"
ZONE_PATH = ROOT / "data/reference/taxi_zone_lookup.csv"

print(f"Repository: {ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Zonen:      {ZONE_PATH}")

In [ ]:
trips = pd.read_parquet(CHECKPOINT_PATH)
zones = pd.read_csv(ZONE_PATH, keep_default_na=False)

required = {
    "ingest_record_id",
    "trip_distance",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "include_demand_after_event_resolution",
}
assert required.issubset(trips.columns)
assert trips["ingest_record_id"].is_unique
assert zones["LocationID"].is_unique

print(f"Checkpoint: {trips.shape}")
print(f"Zonenreferenz: {zones.shape}")

## Decision Log

| Kandidat oder Beobachtung | statistische Evidenz | fachliche Evidenz | Aktion | betroffene Zielsicht | Unsicherheit |
|---|---|---|---|---|---|
| | | | | | |

## 1. Analysepopulation und abgeleitete Merkmale

Entscheidet zuerst, auf welcher Population ihr Schwellen und Modelle bestimmt: vollständiger Audit-Bestand oder vorläufige Nachfragesicht aus Aufgabe 1. Dokumentiert die Wahl und führt bei Bedarf einen Sensitivitätsvergleich durch.

Erzeugt mindestens Fahrtdauer, mittlere Geschwindigkeit, Betrag pro Meile, Stunde und Wochentag. Schützt jede Division gegen null und nichtpositive Werte.

In [ ]:
# Eure Analysepopulation und abgeleiteten Merkmale


## 2. Verteilungen und IQR

Beschreibt Distanz, Dauer, Geschwindigkeit und Betrag mit robusten Kennzahlen und geeigneten Visualisierungen. Vergleicht bei stark rechtsschiefen Verteilungen eine lineare mit einer logarithmischen Darstellung.

\[
IQR = Q_{0.75} - Q_{0.25}
\]

\[
\text{obere Grenze} = Q_{0.75} + 1.5 \cdot IQR
\]

Kennzeichnet die Treffer als Kandidaten. Prüft Stichproben knapp oberhalb der Grenze und aus dem äussersten Rand.

In [ ]:
# Verteilungen, Quantile und Visualisierungen


In [ ]:
# IQR-Grenze bestimmen und Kandidaten kennzeichnen


## 3. Multivariate Anomalien

Trainiert einen Isolation Forest mit erklärbaren Features. Dokumentiert Vorverarbeitung, Modellparameter und `random_state`. Kategoriale Codes sind keine kontinuierlichen Messwerte.

```python
# Beispielgerüst – Parameter und Features müsst ihr begründen:
# model = IsolationForest(
#     n_estimators=...,
#     contamination=...,
#     random_state=...,
# )
```

In [ ]:
# Modellmatrix aufbauen, Isolation Forest trainieren und Scores speichern


In [ ]:
# IQR- und Isolation-Forest-Kandidaten vergleichen


## 4. Zonen- und Tarifevidenz

Verbindet `PULocationID` und `DOLocationID` mit der offiziellen Zonenreferenz. Untersucht, ob Zone, Service-Zone oder Tarifcode einzelne Extremfälle plausibler machen. Ein Lookup liefert Kontext, aber noch keinen Beweis für die reale Fahrt.

In [ ]:
# Zonenreferenz für Pickup und Dropoff anbinden


In [ ]:
# Kompakte Prüftabelle für auffällige Fälle erstellen


## 5. Behalten, ausschliessen oder prüfen

Definiert konservative Regeln für klare physische beziehungsweise zeitliche Fehler. Haltet statistische Kandidaten, valide Extremfälle und mehrdeutige Messungen auseinander.

| Evidenz | Klasse | Demand | Revenue | Duration | Begründung |
|---|---|---|---|---|---|
| | | | | | |

In [ ]:
# Transparente Flags und zielabhängige Entscheidungen


## 6. KI-Experiment

Gebt ChatGPT zunächst nur eine kleine Tabelle numerischer Kandidaten. Wiederholt die Anfrage mit Spaltenbeschreibung, Tarif- und Zoneninformationen, erlaubten Aktionen sowie der Aufforderung, Unsicherheit auszuweisen.

Optional zeigt die Kursleitung, wie ein kontextarmer Online-Workflow zu nicht reproduzierbaren Pfaden, Parametern oder Löschregeln führen kann.

### Dokumentation des KI-Einsatzes

- verwendetes System:
- Kontext der ersten Anfrage:
- erfundene oder nicht belegte Aussagen:
- verbesserter Prompt:
- Prüfungen am erzeugten Code:
- menschliche Fachentscheidung:

In [ ]:
# Optional: KI-Code zuerst an wenigen bekannten Kandidaten testen.


## 7. Kurzpräsentation

Zeigt je mindestens einen validen Extremfall, klaren Noise-Fall und Review-Fall. Trennt dabei Modellscore, fachliche Evidenz und endgültige Aktion.

In [ ]:
# Technische Mindestkontrollen
assert len(trips) > 0
assert trips["ingest_record_id"].is_unique
assert zones["LocationID"].is_unique
print("Mindestkontrollen bestanden. Ergänzt eure fachlichen Prüfungen.")